In [2]:
!pip install -q pypdf

import os
import re
import json
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import pypdf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

# Optional libraries for production vector DB & LLM APIs (gracefully handled if absent)
try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False

print(f"Environment ready | pypdf={pypdf.__version__} | numpy={np.__version__} | FAISS available={HAS_FAISS}")

Environment ready | pypdf=7.1.0 | numpy=2.4.6 | FAISS available=False


In [3]:
PDF_PATH = "/content/book.pdf"

reader = pypdf.PdfReader(PDF_PATH)
raw_pages: List[Dict[str, Any]] = []

for idx, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    raw_pages.append({
        "pdf_page": idx + 1,
        "char_count": len(text),
        "word_count": len(text.split()),
        "raw_text": text
    })

CHAPTER_RANGES = [
    (1, 14, "Front Matter & Preface"),
    (15, 38, "Chapter 1: Introduction"),
    (39, 144, "Chapter 2: Supervised Learning"),
    (145, 224, "Chapter 3: Unsupervised Learning and Preprocessing"),
    (225, 264, "Chapter 4: Representing Data and Engineering Features"),
    (265, 318, "Chapter 5: Model Evaluation and Improvement"),
    (319, 336, "Chapter 6: Algorithm Chains and Pipelines"),
    (337, 370, "Chapter 7: Working with Text Data"),
    (371, 380, "Chapter 8: Wrapping Up"),
    (381, 392, "Back Matter / Index")
]

def get_chapter_for_page(pdf_page: int) -> str:
    for start, end, title in CHAPTER_RANGES:
        if start <= pdf_page <= end:
            return title
    return "Unknown"

def clean_book_text(text: str) -> str:
    # 1. Normalize soft hyphens and join hyphenated line wraps
    text = text.replace("\xad", "").replace("\u2010", "-")
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    # 2. Strip running chapter headers and footers
    text = re.sub(r'\n?\d+\s*\|\s*Chapter\s+\d+:[^\n]*\n?', '\n', text)
    text = re.sub(r'\n?Chapter\s+\d+:[^\n]*\|\s*\d+\n?', '\n', text)
    # 3. Collapse redundant blank lines and horizontal whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

cleaned_pages: List[Dict[str, Any]] = []
low_text_pages: List[int] = []

for p in raw_pages:
    if p["pdf_page"] >= 381:  # Exclude back-of-book Index
        continue
    cleaned = clean_book_text(p["raw_text"])
    if len(cleaned) < 40:
        low_text_pages.append(p["pdf_page"])
        continue
    cleaned_pages.append({
        "pdf_page": p["pdf_page"],
        "chapter": get_chapter_for_page(p["pdf_page"]),
        "text": cleaned
    })

print(f"Loaded PDF: '{PDF_PATH}' (Introduction to Machine Learning with Python — Andreas C. Müller & Sarah Guido)")
print(f"Total Pages: {len(raw_pages)}")
print(f"Total Characters (raw): {sum(p['char_count'] for p in raw_pages):,}")
print(f"Total Words (raw): {sum(p['word_count'] for p in raw_pages):,}")
print(f"Low-text / Blank / Cover pages (<40 chars): {low_text_pages}")
print(f"Excluded Back-of-Book Index pages: PDF pp. 381–392 (12 pages)")
print(f"Active Knowledge Base Pages Retained: {len(cleaned_pages)}")

Loaded PDF: '/content/book.pdf' (Introduction to Machine Learning with Python — Andreas C. Müller & Sarah Guido)
Total Pages: 392
Total Characters (raw): 691,894
Total Words (raw): 103,206
Low-text / Blank / Cover pages (<40 chars): [2, 144, 224, 318, 336]
Excluded Back-of-Book Index pages: PDF pp. 381–392 (12 pages)
Active Knowledge Base Pages Retained: 375


In [4]:
def create_overlapping_chunks(
    pages: List[Dict[str, Any]],
    chunk_size_tokens: int = 650,
    overlap_tokens: int = 100
) -> List[Dict[str, Any]]:
    chunk_words = int(chunk_size_tokens * 0.75)
    overlap_words = int(overlap_tokens * 0.75)
    step = max(1, chunk_words - overlap_words)

    # Stream words with page & chapter provenance
    word_records = []
    for p in pages:
        for word in p["text"].split():
            word_records.append((word, p["pdf_page"], p["chapter"]))

    chunks: List[Dict[str, Any]] = []
    for cid, start in enumerate(range(0, len(word_records), step)):
        window = word_records[start:start + chunk_words]
        if len(window) < 40:
            break
        text = " ".join(w[0] for w in window)
        pages_covered = sorted({w[1] for w in window})
        chapters_covered = list(dict.fromkeys(w[2] for w in window))

        chunks.append({
            "chunk_id": f"chunk_{cid:04d}",
            "page_start": pages_covered[0],
            "page_end": pages_covered[-1],
            "chapter": chapters_covered[0],
            "word_count": len(window),
            "est_tokens": int(round(len(window) / 0.75)),
            "text": text
        })
        if start + chunk_words >= len(word_records):
            break
    return chunks

chunks = create_overlapping_chunks(cleaned_pages, chunk_size_tokens=650, overlap_tokens=100)

print(f"Chunking Configuration: chunk_size = 650 tokens (~487 words), overlap = 100 tokens (~75 words)")
print(f"Total Chunks Created: {len(chunks)}")
print(f"Average Chunk Size: {np.mean([c['word_count'] for c in chunks]):.1f} words (~{np.mean([c['est_tokens'] for c in chunks]):.1f} tokens)")
print(f"Min / Max Chunk Tokens: {min(c['est_tokens'] for c in chunks)} / {max(c['est_tokens'] for c in chunks)} tokens\n")
print(f"Sample Chunk [{chunks[0]['chunk_id']}]:")
print(f"  Location: PDF pp. {chunks[0]['page_start']}–{chunks[0]['page_end']} | {chunks[0]['chapter']}")
print(f"  Preview : {chunks[0]['text'][:280]}...")

Chunking Configuration: chunk_size = 650 tokens (~487 words), overlap = 100 tokens (~75 words)
Total Chunks Created: 238
Average Chunk Size: 485.5 words (~646.9 tokens)
Min / Max Chunk Tokens: 161 / 649 tokens

Sample Chunk [chunk_0000]:
  Location: PDF pp. 1–5 | Front Matter & Preface
  Preview : Andreas C. Müller & Sarah Guido Introduction to Machine Learning with P y thon A GUIDE FOR DATA SCIENTISTS Andreas C. Müller and Sarah Guido Introduction to Machine Learning with Python A Guide for Data Scientists Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol T...


In [5]:
class HybridVectorStore:
    """Vector database supporting dense semantic + sparse lexical cosine similarity search."""

    def __init__(self, n_components: int = 220, max_features: int = 25000, semantic_weight: float = 0.65):
        self.semantic_weight = semantic_weight
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            stop_words="english",
            ngram_range=(1, 2),
            sublinear_tf=True
        )
        self.svd = TruncatedSVD(n_components=n_components, random_state=42)
        self.dense_vectors: Optional[np.ndarray] = None
        self.sparse_vectors = None
        self.faiss_index = None
        self.chunks: List[Dict[str, Any]] = []

    def add_chunks(self, chunks: List[Dict[str, Any]]) -> None:
        self.chunks = chunks
        texts = [c["text"] for c in chunks]
        tfidf_mat = self.vectorizer.fit_transform(texts)
        self.sparse_vectors = normalize(tfidf_mat, norm="l2")

        effective_dim = min(self.svd.n_components, tfidf_mat.shape[0] - 1)
        self.svd.n_components = effective_dim
        dense_raw = self.svd.fit_transform(tfidf_mat)
        self.dense_vectors = normalize(dense_raw, norm="l2").astype("float32")

        if HAS_FAISS:
            self.faiss_index = faiss.IndexFlatIP(self.dense_vectors.shape[1])
            self.faiss_index.add(self.dense_vectors)

    def search(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        q_sparse = normalize(self.vectorizer.transform([query]), norm="l2")
        q_dense = normalize(self.svd.transform(q_sparse), norm="l2").astype("float32")

        if self.faiss_index is not None:
            dense_sim, _ = self.faiss_index.search(q_dense, len(self.chunks))
            # Recompute full vector dot product to align indices cleanly
            dense_sim = (self.dense_vectors @ q_dense[0]).ravel()
        else:
            dense_sim = (self.dense_vectors @ q_dense[0]).ravel()

        sparse_sim = (self.sparse_vectors @ q_sparse.T).toarray().ravel()
        combined = self.semantic_weight * dense_sim + (1.0 - self.semantic_weight) * sparse_sim

        top_indices = np.argsort(combined)[::-1][:top_k]
        results = []
        for rank, idx in enumerate(top_indices, start=1):
            c = self.chunks[idx]
            results.append({
                "rank": rank,
                "chunk_id": c["chunk_id"],
                "score": round(float(combined[idx]), 4),
                "dense_score": round(float(dense_sim[idx]), 4),
                "lexical_score": round(float(sparse_sim[idx]), 4),
                "pages": f"PDF pp. {c['page_start']}–{c['page_end']}",
                "chapter": c["chapter"],
                "text": c["text"]
            })
        return results

# Build and populate the vector store
vector_db = HybridVectorStore(n_components=220, max_features=25000, semantic_weight=0.65)
vector_db.add_chunks(chunks)

# Demonstrate similarity search on a sample query
sample_query = "What is the difference between Ridge and Lasso regression?"
sample_results = vector_db.search(sample_query, top_k=3)

print("Vector Store Built Successfully!")
print(f"  - Indexed Chunks : {len(vector_db.chunks)}")
print(f"  - Dense Matrix   : {vector_db.dense_vectors.shape} (L2-normalized LSA/SVD semantic embeddings, dtype={vector_db.dense_vectors.dtype})")
print(f"  - Sparse Matrix  : {vector_db.sparse_vectors.shape} (Sublinear TF-IDF unigram+bigram vectors)")
print(f"\nSample Query: '{sample_query}'")
print("Top-3 Retrieved Chunks:")
for r in sample_results:
    print(f"  [{r['rank']}] {r['chunk_id']} | Score: {r['score']:.4f} (Semantic: {r['dense_score']:.4f}, Lexical: {r['lexical_score']:.4f}) | {r['pages']} | {r['chapter']}")
    print(f"      Text: {r['text'][:230]}...")

Vector Store Built Successfully!
  - Indexed Chunks : 238
  - Dense Matrix   : (238, 220) (L2-normalized LSA/SVD semantic embeddings, dtype=float32)
  - Sparse Matrix  : (238, 25000) (Sublinear TF-IDF unigram+bigram vectors)

Sample Query: 'What is the difference between Ridge and Lasso regression?'
Top-3 Retrieved Chunks:
  [1] chunk_0043 | Score: 0.5791 (Semantic: 0.7932, Lexical: 0.1813) | PDF pp. 66–68 | Chapter 2: Supervised Learning
      Text: for ridge is better, particularly for small subsets of the data. For less than 400 data points, linear regression is not able to learn anything. As more and more data becomes available to the model, both models improve, and linear...
  [2] chunk_0044 | Score: 0.4749 (Semantic: 0.6507, Lexical: 0.1485) | PDF pp. 68–70 | Chapter 2: Supervised Learning
      Text: max_iter=100000).fit(X_train, y_train) print("Training set score: {:.2f}".format(lasso001.score(X_train, y_train))) print("Test set score: {:.2f}".format(lasso001.score(X_test, y_te

In [6]:
def retrieve_context(query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """Accept a user query, embed it, and retrieve top-k relevant book chunks."""
    return vector_db.search(query=query, top_k=top_k)

test_query_k = "How does Stratified k-Fold cross-validation work in scikit-learn?"
print(f"Top-k Retrieval Experimentation for Query:\n  '{test_query_k}'\n")

for k_val in [3, 5, 7]:
    retrieved_k = retrieve_context(test_query_k, top_k=k_val)
    mean_score = np.mean([item["score"] for item in retrieved_k])
    est_tokens = sum(int(round(len(item["text"].split()) / 0.75)) for item in retrieved_k)
    chapters = list(dict.fromkeys(item["chapter"] for item in retrieved_k))
    pages = [item["pages"] for item in retrieved_k]

    print(f"k = {k_val} | Mean Score: {mean_score:.4f} (Top-1: {retrieved_k[0]['score']:.4f}, Rank-{k_val}: {retrieved_k[-1]['score']:.4f}) | Est. Context: ~{est_tokens:,} tokens")
    print(f"   Chapters : {chapters}")
    print(f"   Pages    : {pages}\n")

Top-k Retrieval Experimentation for Query:
  'How does Stratified k-Fold cross-validation work in scikit-learn?'

k = 3 | Mean Score: 0.4544 (Top-1: 0.6215, Rank-3: 0.3272) | Est. Context: ~1,947 tokens
   Chapters : ['Chapter 5: Model Evaluation and Improvement']
   Pages    : ['PDF pp. 269–270', 'PDF pp. 268–269', 'PDF pp. 265–267']

k = 5 | Mean Score: 0.4005 (Top-1: 0.6215, Rank-5: 0.3184) | Est. Context: ~3,245 tokens
   Chapters : ['Chapter 5: Model Evaluation and Improvement']
   Pages    : ['PDF pp. 269–270', 'PDF pp. 268–269', 'PDF pp. 265–267', 'PDF pp. 270–272', 'PDF pp. 266–268']

k = 7 | Mean Score: 0.3636 (Top-1: 0.6215, Rank-7: 0.2682) | Est. Context: ~4,543 tokens
   Chapters : ['Chapter 5: Model Evaluation and Improvement']
   Pages    : ['PDF pp. 269–270', 'PDF pp. 268–269', 'PDF pp. 265–267', 'PDF pp. 270–272', 'PDF pp. 266–268', 'PDF pp. 278–280', 'PDF pp. 286–287']



In [7]:
STRICT_RAG_SYSTEM_PROMPT = """You are a rigorous Machine Learning Knowledge Assistant answering questions strictly from the textbook
'Introduction to Machine Learning with Python' by Andreas C. Müller & Sarah Guido.

STRICT GROUNDING RULES:
1. Base your answer EXCLUSIVELY on the provided <retrieved_context> passages.
2. Do NOT use outside knowledge, assumptions, or unstated facts.
3. Cite supporting passages inline using their exact [chunk_id | PDF pp. X–Y] tags.
4. Preserve exact scikit-learn class names, parameter names, and code conventions from the context.
5. If the <retrieved_context> does not contain enough information to answer the question, respond ONLY with:
   "I could not find any information about this topic in the retrieved sections of the book."
"""

def build_rag_prompt(query: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    context_blocks = []
    for c in retrieved_chunks:
        header = f"[{c['chunk_id']} | {c['chapter']} | {c['pages']} | score={c['score']:.4f}]"
        context_blocks.append(f"{header}\n{c['text']}")
    formatted_context = "\n\n---\n\n".join(context_blocks)

    return (
        f"{STRICT_RAG_SYSTEM_PROMPT}\n\n"
        f"<retrieved_context>\n{formatted_context}\n</retrieved_context>\n\n"
        f"User Question: {query}\n"
        f"Grounded Answer:"
    )

class GroundedRAGGenerator:
    """Generates strictly grounded answers using Gemini/OpenAI APIs or an offline extractive synthesizer."""

    def __init__(self, min_confidence: float = 0.15):
        self.min_confidence = min_confidence

    def generate(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Guardrail 1: Abstain if retrieval confidence is below threshold
        if not retrieved_chunks or retrieved_chunks[0]["score"] < self.min_confidence:
            return {
                "answer": (
                    "I could not find any information about this topic in the retrieved sections of "
                    "Introduction to Machine Learning with Python. In accordance with strict context "
                    "grounding, this topic is outside the scope of the indexed textbook."
                ),
                "grounded": False,
                "backend": "confidence_guardrail",
                "citations": []
            }

        prompt = build_rag_prompt(query, retrieved_chunks)

        # Option A: Google Gemini API (if GEMINI_API_KEY is set)
        if os.getenv("GEMINI_API_KEY"):
            try:
                from google import genai
                client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
                resp = client.models.generate_content(
                    model="gemini-2.5-flash",
                    contents=prompt,
                    config={"temperature": 0.0}
                )
                return {
                    "answer": resp.text.strip(),
                    "grounded": True,
                    "backend": "gemini-2.5-flash",
                    "citations": [f"{c['chunk_id']} ({c['pages']})" for c in retrieved_chunks[:3]]
                }
            except Exception:
                pass

        # Option B: OpenAI API (if OPENAI_API_KEY is set)
        if os.getenv("OPENAI_API_KEY"):
            try:
                from openai import OpenAI
                client = OpenAI()
                resp = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0
                )
                return {
                    "answer": resp.choices[0].message.content.strip(),
                    "grounded": True,
                    "backend": "gpt-4o-mini",
                    "citations": [f"{c['chunk_id']} ({c['pages']})" for c in retrieved_chunks[:3]]
                }
            except Exception:
                pass

        # Option C: Deterministic Offline Extractive-Grounded Synthesizer
        return self._extractive_grounded_synthesis(query, retrieved_chunks)

    def _extractive_grounded_synthesis(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Score individual sentences from top retrieved chunks against the query
        sentences_meta = []
        for c in retrieved_chunks[:4]:
            raw_sents = re.split(r'(?<=[.!?])\s+', c["text"])
            for s in raw_sents:
                s_clean = s.strip()
                if 45 <= len(s_clean) <= 420 and not s_clean.startswith("In["):
                    sentences_meta.append((s_clean, c["chunk_id"], c["pages"], c["chapter"]))

        if not sentences_meta:
            top = retrieved_chunks[0]
            return {
                "answer": f"From {top['chapter']} ({top['pages']}): {top['text'][:500]}...",
                "grounded": True,
                "backend": "offline_extractive_grounded",
                "citations": [f"{top['chunk_id']} ({top['pages']})"]
            }

        sent_texts = [x[0] for x in sentences_meta]
        sent_vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
        S_mat = sent_vec.fit_transform(sent_texts)
        q_vec = sent_vec.transform([query])
        sims = (S_mat @ q_vec.T).toarray().ravel()

        # Select top-4 non-redundant sentences preserving source citations
        ranked_idx = np.argsort(sims)[::-1]
        selected = []
        seen_prefixes = set()
        for idx in ranked_idx:
            if sims[idx] <= 0.02 and len(selected) >= 2:
                break
            sent, cid, pages, chap = sentences_meta[idx]
            prefix = sent[:50].lower()
            if prefix in seen_prefixes:
                continue
            seen_prefixes.add(prefix)
            selected.append(f"- {sent} *[{cid}, {chap}, {pages}]*")
            if len(selected) == 4:
                break

        citations = list(dict.fromkeys([f"{c['chunk_id']} ({c['pages']})" for c in retrieved_chunks[:3]]))
        answer_text = (
            "Based on the retrieved passages from *Introduction to Machine Learning with Python*:\n\n"
            + "\n".join(selected)
        )
        return {
            "answer": answer_text,
            "grounded": True,
            "backend": "offline_extractive_grounded",
            "citations": citations
        }

rag_generator = GroundedRAGGenerator(min_confidence=0.15)
print("GroundedRAGGenerator initialized with strict anti-hallucination prompt & confidence guardrail.")

GroundedRAGGenerator initialized with strict anti-hallucination prompt & confidence guardrail.


In [8]:
class MLKnowledgeAssistant:
    """End-to-End RAG Machine Learning Knowledge Assistant over book.pdf."""

    def __init__(self, pdf_path: str = "book.pdf", chunk_size_tokens: int = 650, overlap_tokens: int = 100):
        self.pdf_path = pdf_path
        self.chunk_size_tokens = chunk_size_tokens
        self.overlap_tokens = overlap_tokens
        self.vector_store = HybridVectorStore(n_components=220, max_features=25000, semantic_weight=0.65)
        self.generator = GroundedRAGGenerator(min_confidence=0.15)
        self.is_indexed = False

    def build_knowledge_base(self) -> Dict[str, Any]:
        reader = pypdf.PdfReader(self.pdf_path)
        pages = []
        for idx, page in enumerate(reader.pages):
            pdf_page = idx + 1
            if pdf_page >= 381:
                continue
            cleaned = clean_book_text(page.extract_text() or "")
            if len(cleaned) >= 40:
                pages.append({
                    "pdf_page": pdf_page,
                    "chapter": get_chapter_for_page(pdf_page),
                    "text": cleaned
                })
        doc_chunks = create_overlapping_chunks(pages, self.chunk_size_tokens, self.overlap_tokens)
        self.vector_store.add_chunks(doc_chunks)
        self.is_indexed = True
        return {"active_pages": len(pages), "total_chunks": len(doc_chunks)}

    def ask(self, question: str, top_k: int = 5) -> Dict[str, Any]:
        if not self.is_indexed:
            self.build_knowledge_base()
        retrieved = self.vector_store.search(question, top_k=top_k)
        gen_result = self.generator.generate(question, retrieved)
        return {
            "question": question,
            "top_k": top_k,
            "retrieved_chunks": retrieved,
            "answer": gen_result["answer"],
            "grounded": gen_result["grounded"],
            "backend": gen_result["backend"],
            "citations": gen_result["citations"]
        }

# Instantiate and run end-to-end demonstration across 5 queries
assistant = MLKnowledgeAssistant(pdf_path="book.pdf")
stats = assistant.build_knowledge_base()

demo_queries = [
    "What is the difference between Ridge and Lasso regression in scikit-learn?",
    "How does Stratified k-Fold cross-validation work in scikit-learn?",
    "How do Pipelines in scikit-learn prevent data leakage during GridSearchCV?",
    "What is TF-IDF rescaling and how does TfidfVectorizer work on text data?",
    "How does the book define quantum reinforcement learning on Mars?"
]

for i, q in enumerate(demo_queries, start=1):
    res = assistant.ask(q, top_k=5)
    top_ev = ", ".join([f"{c['chunk_id']} ({c['pages']}, score={c['score']:.4f})" for c in res["retrieved_chunks"][:3]])
    print("=" * 85)
    print(f"QUERY {i}: {res['question']}")
    print(f"TOP RETRIEVED EVIDENCE: {top_ev}")
    print("-" * 85)
    print(f"GROUNDED ASSISTANT RESPONSE (grounded={res['grounded']} | backend={res['backend']}):\n{res['answer']}\n")

QUERY 1: What is the difference between Ridge and Lasso regression in scikit-learn?
TOP RETRIEVED EVIDENCE: chunk_0043 (PDF pp. 66–68, score=0.5264), chunk_0044 (PDF pp. 68–70, score=0.4513), chunk_0042 (PDF pp. 64–67, score=0.3084)
-------------------------------------------------------------------------------------
GROUNDED ASSISTANT RESPONSE (grounded=True | backend=offline_extractive_grounded):
Based on the retrieved passages from *Introduction to Machine Learning with Python*:

- scikit-learn also provides the ElasticNet class, which combines the penalties of Lasso and Ridge. *[chunk_0044, Chapter 2: Supervised Learning, PDF pp. 68–70]*
- scikit-learn always stores anything that is derived from the training data in attributes that end with a trailing underscore. *[chunk_0040, Chapter 2: Supervised Learning, PDF pp. 61–63]*
- Comparing coefficient magnitudes for lasso regression with different values of alpha and ridge regression For alpha=1, we not only see that most of the coeffi

In [9]:
def interactive_qa_session(assistant: MLKnowledgeAssistant, top_k: int = 5):
    """Interactive CLI loop for real-time question answering over book.pdf."""
    print("ML Knowledge Assistant Ready! Type your ML question (or 'exit' to quit).\n")
    while True:
        user_q = input("Ask a question about ML with Python: ").strip()
        if user_q.lower() in {"exit", "quit", "q", ""}:
            print("Session ended.")
            break
        result = assistant.ask(user_q, top_k=top_k)
        print(f"\n[Sources: {', '.join(result['citations'])}]")
        print(result["answer"])
        print("-" * 80)

# Example single-turn real-time call:
live_demo = assistant.ask("When should I use Random Forests versus Gradient Boosted Decision Trees?", top_k=5)
print("Live Real-Time Query:", live_demo["question"])
print("Citations:", live_demo["citations"])
print("\n" + live_demo["answer"])

Live Real-Time Query: When should I use Random Forests versus Gradient Boosted Decision Trees?
Citations: ['chunk_0087 (PDF pp. 141–145)', 'chunk_0067 (PDF pp. 104–106)', 'chunk_0061 (PDF pp. 97–98)']

Based on the retrieved passages from *Introduction to Machine Learning with Python*:

- Gradient boosted decision trees Often slightly more accurate than random forests. *[chunk_0087, Chapter 2: Supervised Learning, PDF pp. 141–145]*
- Gradient boosted decision trees are among the most powerful and widely used models for supervised learning. *[chunk_0067, Chapter 2: Supervised Learning, PDF pp. 104–106]*
- There are many models in the machine learning literature that belong to this category, but there are two ensemble models that have proven to be effective on a wide range of datasets for classification and regression, both of which use decision trees as their building blocks: random forests and gradient boosted decision trees. *[chunk_0061, Chapter 2: Supervised Learning, PDF pp. 97–98]